# ✂️ Hashformers Benchmark: Word Segmentation Library Comparison

This notebook benchmarks **hashformers** against various word segmentation approaches:

| Category | Libraries |
|----------|-----------|
| Classic Statistical | `wordninja`, `symspellpy` |
| Social Media Specialist | `ekphrasis` |
| Modern LLMs | `Qwen2-0.5B-Instruct` (4-bit quantized) |
| Hashformers | `gpt2` (baseline), `distilgpt2` (no reranker) |

**Benchmark Data:** 20 samples from each of 10 datasets (200 total):
- Hashtag datasets: `stan_small`, `stan_large_*`, `Test-Stanford`
- Identifier datasets: `binkley`, `bt11`, `jhotdraw`, `lynx`, `loyola-udelaware`

**Requirements:** Google Colab with GPU runtime (T4 recommended)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruanchaves/hashformers/blob/master/benchmark_notebook.ipynb)


## 1. Environment Setup

Install all dependencies and download required corpus files.


In [ ]:
# ============================================================================
# CELL 1: Environment Setup
# ============================================================================

# Install all required packages
# Install hashformers from the benchmark branch (not PyPI)
!pip install -q git+https://github.com/ruanchaves/hashformers.git@benchmark
!pip install -q wordninja symspellpy ekphrasis transformers accelerate bitsandbytes scipy pandas matplotlib seaborn

# Download SymSpell frequency dictionary
!wget -q -nc https://raw.githubusercontent.com/mammothb/symspellpy/master/symspellpy/frequency_dictionary_en_82_765.txt

# Clone the hashformers repo to access datasets (if running on Colab)
import os
if not os.path.exists("datasets"):
    !git clone -q --depth 1 https://github.com/ruanchaves/hashformers.git temp_repo
    !mv temp_repo/datasets datasets
    !rm -rf temp_repo

# Trigger Ekphrasis corpus download
print("Downloading Ekphrasis corpora...")
from ekphrasis.classes.preprocessor import TextPreProcessor
from ekphrasis.classes.tokenizer import SocialTokenizer
from ekphrasis.dicts.emoticons import emoticons

# This initialization triggers the download of Twitter corpus files
_ekphrasis_init = TextPreProcessor(
    normalize=['url', 'email', 'percent', 'money', 'phone', 'user', 'time', 'date', 'number'],
    segmenter="twitter",
    corrector="twitter",
    unpack_hashtags=True,
    tokenizer=SocialTokenizer(lowercase=True).tokenize,
)
del _ekphrasis_init

print("✅ Environment setup complete!")


## 2. Segmenter Architecture

Unified interface for all word segmentation tools with concrete implementations.


In [ ]:
# ============================================================================
# CELL 2: Segmenter Architecture
# ============================================================================

from abc import ABC, abstractmethod
from typing import Optional
import re


class Segmenter(ABC):
    """Abstract base class for word segmentation tools."""
    
    @abstractmethod
    def segment(self, text: str) -> str:
        """
        Segment a hashtag or concatenated string into space-separated words.
        
        Args:
            text: Input text (hashtag without # symbol)
            
        Returns:
            Space-separated segmented text
        """
        pass
    
    def _clean_input(self, text: str) -> str:
        """Remove # symbol and clean input text."""
        return text.lstrip("#").strip()


# -----------------------------------------------------------------------------
# WordNinja Segmenter
# -----------------------------------------------------------------------------
import wordninja


class WordNinjaSegmenter(Segmenter):
    """Word segmentation using WordNinja (statistical n-gram model)."""
    
    def __init__(self):
        """Initialize WordNinja segmenter."""
        # WordNinja loads its model on first use
        pass
    
    def segment(self, text: str) -> str:
        """Segment text using WordNinja."""
        cleaned = self._clean_input(text)
        words = wordninja.split(cleaned)
        return " ".join(words)


# -----------------------------------------------------------------------------
# SymSpell Segmenter
# -----------------------------------------------------------------------------
from symspellpy import SymSpell, Verbosity


class SymSpellSegmenter(Segmenter):
    """Word segmentation using SymSpell (symmetric delete spelling correction)."""
    
    def __init__(self, dictionary_path: str = "frequency_dictionary_en_82_765.txt"):
        """
        Initialize SymSpell with frequency dictionary.
        
        Args:
            dictionary_path: Path to the frequency dictionary file
        """
        self.sym_spell = SymSpell(max_dictionary_edit_distance=0, prefix_length=7)
        if not self.sym_spell.load_dictionary(
            dictionary_path, 
            term_index=0, 
            count_index=1
        ):
            raise FileNotFoundError(f"Dictionary not found: {dictionary_path}")
    
    def segment(self, text: str) -> str:
        """Segment text using SymSpell word segmentation."""
        cleaned = self._clean_input(text).lower()
        result = self.sym_spell.word_segmentation(cleaned)
        return result.corrected_string


# -----------------------------------------------------------------------------
# Ekphrasis Segmenter
# -----------------------------------------------------------------------------
from ekphrasis.classes.preprocessor import TextPreProcessor
from ekphrasis.classes.tokenizer import SocialTokenizer
from ekphrasis.dicts.emoticons import emoticons


class EkphrasisSegmenter(Segmenter):
    """Word segmentation using Ekphrasis (social media text processor)."""
    
    def __init__(self, corpus: str = "twitter"):
        """
        Initialize Ekphrasis text processor.
        
        Args:
            corpus: Corpus for word statistics ('twitter' or 'english')
        """
        self.text_processor = TextPreProcessor(
            normalize=['url', 'email', 'percent', 'money', 'phone', 'user',
                      'time', 'date', 'number'],
            annotate={"hashtag", "allcaps", "elongated", "repeated",
                     'emphasis', 'censored'},
            fix_html=True,
            segmenter=corpus,
            corrector=corpus,
            unpack_hashtags=True,
            unpack_contractions=True,
            spell_correct_elong=False,
            tokenizer=SocialTokenizer(lowercase=True).tokenize,
            dicts=[emoticons]
        )
    
    def segment(self, text: str) -> str:
        """Segment text using Ekphrasis."""
        cleaned = self._clean_input(text)
        # Ekphrasis expects hashtag with # symbol
        tokens = self.text_processor.pre_process_doc("#" + cleaned)
        # Remove the <hashtag> and </hashtag> annotation tokens
        tokens = [t for t in tokens if not t.startswith("<") and not t.endswith(">")]
        return " ".join(tokens)


# -----------------------------------------------------------------------------
# Hashformers Segmenter
# -----------------------------------------------------------------------------
from hashformers import TransformerWordSegmenter


class HashformersSegmenter(Segmenter):
    """Word segmentation using Hashformers (Transformer beam search)."""
    
    def __init__(
        self,
        segmenter_model: str = "gpt2",
        segmenter_type: str = "incremental",
        reranker_model: Optional[str] = None,
        reranker_type: Optional[str] = None
    ):
        """
        Initialize Hashformers word segmenter.
        
        Args:
            segmenter_model: HuggingFace model name for segmentation
            segmenter_type: Model type ('incremental', 'masked', or 'seq2seq')
            reranker_model: Optional reranker model name
            reranker_type: Optional reranker model type
        """
        self.ws = TransformerWordSegmenter(
            segmenter_model_name_or_path=segmenter_model,
            segmenter_model_type=segmenter_type,
            reranker_model_name_or_path=reranker_model,
            reranker_model_type=reranker_type
        )
        self.model_name = segmenter_model
    
    def segment(self, text: str) -> str:
        """Segment text using Hashformers."""
        cleaned = self._clean_input(text)
        results = self.ws.segment([cleaned])
        return results[0] if results else cleaned


# -----------------------------------------------------------------------------
# Local LLM Segmenter (using Qwen2-0.5B - small, stable, instruction-tuned)
# -----------------------------------------------------------------------------
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


class LocalLLMSegmenter(Segmenter):
    """Word segmentation using a local quantized LLM with prompting."""
    
    def __init__(
        self,
        model_name: str = "Qwen/Qwen2-0.5B-Instruct",
        load_in_4bit: bool = True,
        max_new_tokens: int = 64
    ):
        """
        Initialize local LLM for segmentation.
        
        Args:
            model_name: HuggingFace model name
            load_in_4bit: Whether to use 4-bit quantization
            max_new_tokens: Maximum tokens to generate
        """
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        
        # Configure quantization
        if load_in_4bit and torch.cuda.is_available():
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True
            )
        else:
            bnb_config = None
        
        # Load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )
        
        # Set pad token if not set
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
    
    def segment(self, text: str) -> str:
        """Segment text using LLM prompting."""
        cleaned = self._clean_input(text)
        
        # Construct messages for Qwen2 chat format with few-shot examples
        messages = [
            {"role": "system", "content": "You split concatenated hashtag words into separate words. Reply with ONLY the space-separated words, nothing else."},
            # Few-shot examples to guide the model
            {"role": "user", "content": "Split: icecream"},
            {"role": "assistant", "content": "ice cream"},
            {"role": "user", "content": "Split: newyorkcity"},
            {"role": "assistant", "content": "new york city"},
            {"role": "user", "content": "Split: machinelearning"},
            {"role": "assistant", "content": "machine learning"},
            {"role": "user", "content": "Split: throwbackthursday"},
            {"role": "assistant", "content": "throwback thursday"},
            {"role": "user", "content": "Split: gameofthrones"},
            {"role": "assistant", "content": "game of thrones"},
            # Actual query
            {"role": "user", "content": f"Split: {cleaned}"}
        ]
        
        # Apply chat template
        prompt = self.tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
        
        # Tokenize and generate
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        # Decode only the new tokens (exclude prompt)
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        answer = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        
        # Clean up: take first line, remove quotes and extra whitespace
        answer = answer.split("\n")[0].strip()
        answer = answer.strip('"\'')
        
        # Remove common prefixes the model might add
        for prefix in ["Answer:", "Result:", "Output:", "Split:"]:
            if answer.lower().startswith(prefix.lower()):
                answer = answer[len(prefix):].strip()
        
        # CRITICAL: Only keep characters from the original input + spaces
        # This prevents the LLM from hallucinating new characters
        cleaned_lower = cleaned.lower()
        filtered_chars = []
        input_idx = 0
        for char in answer.lower():
            if char == ' ':
                # Keep spaces (this is the segmentation)
                filtered_chars.append(' ')
            elif input_idx < len(cleaned_lower) and char == cleaned_lower[input_idx]:
                # Keep characters that match the original input in order
                filtered_chars.append(cleaned[input_idx])  # Preserve original case
                input_idx += 1
            # Skip any hallucinated characters
        
        # If we didn't consume all input characters, the answer is malformed
        if input_idx < len(cleaned):
            # Try a simpler approach: just extract space positions from the answer
            # and insert them into the original input
            answer_no_space = answer.replace(" ", "").lower()
            if answer_no_space == cleaned_lower:
                # Characters match, just different spacing - reconstruct with spaces
                result = []
                orig_idx = 0
                for char in answer.lower():
                    if char == ' ':
                        result.append(' ')
                    else:
                        if orig_idx < len(cleaned):
                            result.append(cleaned[orig_idx])
                            orig_idx += 1
                answer = ''.join(result)
            else:
                # Can't salvage - return original
                return cleaned
        else:
            answer = ''.join(filtered_chars)
        
        # Clean up multiple spaces
        answer = ' '.join(answer.split())
        
        # Fallback: if answer is empty or looks wrong, return cleaned input
        if not answer or answer.lower().replace(" ", "") != cleaned.lower():
            return cleaned
        
        return answer


# -----------------------------------------------------------------------------
# Utility: GPU Memory Management
# -----------------------------------------------------------------------------
import gc


def clear_gpu_memory():
    """Clear GPU memory cache."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory cleared. Current allocation: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


print("✅ Segmenter classes defined!")


## 3. Benchmark Data

Load 20 samples from each dataset in the repository (hashtags and code identifiers).


In [ ]:
# ============================================================================
# CELL 3: Benchmark Data - Load from Multiple Datasets
# ============================================================================

import pandas as pd
import ast
import random

random.seed(42)  # For reproducibility

SAMPLES_PER_DATASET = 20


def load_stanford_csv(filepath: str, n_samples: int = SAMPLES_PER_DATASET) -> list[dict]:
    """Load Stanford-format CSV (stan_small, stan_large_*)."""
    df = pd.read_csv(filepath)
    # Sample randomly
    if len(df) > n_samples:
        df = df.sample(n=n_samples, random_state=42)
    
    results = []
    for _, row in df.iterrows():
        hashtag = row["hashtags"]
        # Parse goldtruths (stored as string list)
        try:
            golds = ast.literal_eval(row["goldtruths"])
            gold = golds[0] if isinstance(golds, list) else str(golds)
        except:
            gold = str(row["goldtruths"])
        results.append({"hashtag": hashtag, "gold": gold, "source": filepath.split("/")[-1]})
    return results


def load_binkley_bt11_csv(filepath: str, n_samples: int = SAMPLES_PER_DATASET) -> list[dict]:
    """Load Binkley/BT11 format CSV (identifier,segmentation-with-dashes,...)."""
    df = pd.read_csv(filepath, header=None)
    # Sample randomly
    if len(df) > n_samples:
        df = df.sample(n=n_samples, random_state=42)
    
    results = []
    for _, row in df.iterrows():
        identifier = str(row[0])  # e.g., "init_g16_i"
        segmented = str(row[1])   # e.g., "init-_-g-16-_-i"
        # Convert dash-separated to space-separated, preserving underscores as single chars
        # The format uses "-" as word boundary and "_" as literal underscore
        gold = segmented.replace("-_-", " _ ").replace("-", " ").strip()
        # Clean up: "init _ g 16 _ i" -> normalize spaces
        gold = " ".join(gold.split())
        results.append({"hashtag": identifier, "gold": gold, "source": filepath.split("/")[-1]})
    return results


def load_colon_separated_txt(filepath: str, n_samples: int = SAMPLES_PER_DATASET) -> list[dict]:
    """Load colon-separated format (jhotdraw.txt, lynx.txt): 'input: gold segmentation'."""
    results = []
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if ":" in line and line.strip()]
    
    # Sample randomly
    if len(lines) > n_samples:
        lines = random.sample(lines, n_samples)
    
    for line in lines:
        parts = line.split(":", 1)
        if len(parts) == 2:
            identifier = parts[0].strip()
            gold = parts[1].strip()
            if identifier and gold:
                results.append({"hashtag": identifier, "gold": gold, "source": filepath.split("/")[-1]})
    return results


def load_loyola_txt(filepath: str, n_samples: int = SAMPLES_PER_DATASET) -> list[dict]:
    """Load Loyola-UDelaware format: space-separated with identifier in col 1, segmentation in col 4."""
    results = []
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
    
    # Sample randomly
    if len(lines) > n_samples:
        lines = random.sample(lines, n_samples)
    
    for line in lines:
        parts = line.split()
        if len(parts) >= 5:
            # Column 1 has ::identifier or identifier
            identifier = parts[1].lstrip(":")
            # Column 4 has dash-separated segmentation
            segmented = parts[4]
            gold = segmented.replace("-", " ").strip()
            if identifier and gold:
                results.append({"hashtag": identifier, "gold": gold, "source": filepath.split("/")[-1]})
    return results


def load_stanford_test_txt(filepath: str, n_samples: int = SAMPLES_PER_DATASET) -> list[dict]:
    """Load Test-Stanford.txt: tab-separated with label=1 for correct segmentation."""
    results = {}
    with open(filepath, "r", encoding="utf-8") as f:
        next(f)  # Skip header
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) >= 4:
                hashtag = parts[1].strip("'")
                segmentation = parts[2].strip("'")
                label = parts[3].strip()
                # Only keep correct segmentations (label=1)
                if label == "1" and hashtag not in results:
                    results[hashtag] = segmentation
    
    # Convert to list and sample
    all_items = [{"hashtag": k, "gold": v, "source": filepath.split("/")[-1]} for k, v in results.items()]
    if len(all_items) > n_samples:
        all_items = random.sample(all_items, n_samples)
    return all_items


# Load from all datasets
print("📂 Loading datasets...")
print("=" * 60)

all_samples = []

# Stanford CSV files
for csv_file in ["stan_small.csv", "stan_large_test.csv", "stan_large_dev.csv", "stan_large_train.csv"]:
    try:
        samples = load_stanford_csv(f"datasets/{csv_file}")
        all_samples.extend(samples)
        print(f"✅ {csv_file}: {len(samples)} samples")
    except Exception as e:
        print(f"⚠️ {csv_file}: Failed - {e}")

# Binkley/BT11 CSV files
for csv_file in ["binkley.csv", "bt11.csv"]:
    try:
        samples = load_binkley_bt11_csv(f"datasets/{csv_file}")
        all_samples.extend(samples)
        print(f"✅ {csv_file}: {len(samples)} samples")
    except Exception as e:
        print(f"⚠️ {csv_file}: Failed - {e}")

# Colon-separated TXT files
for txt_file in ["jhotdraw.txt", "lynx.txt"]:
    try:
        samples = load_colon_separated_txt(f"datasets/{txt_file}")
        all_samples.extend(samples)
        print(f"✅ {txt_file}: {len(samples)} samples")
    except Exception as e:
        print(f"⚠️ {txt_file}: Failed - {e}")

# Loyola-UDelaware
try:
    samples = load_loyola_txt("datasets/loyola-udelaware-identifier-splitting-oracle.txt")
    all_samples.extend(samples)
    print(f"✅ loyola-udelaware-identifier-splitting-oracle.txt: {len(samples)} samples")
except Exception as e:
    print(f"⚠️ loyola-udelaware: Failed - {e}")

# Test-Stanford.txt
try:
    samples = load_stanford_test_txt("datasets/Test-Stanford.txt")
    all_samples.extend(samples)
    print(f"✅ Test-Stanford.txt: {len(samples)} samples")
except Exception as e:
    print(f"⚠️ Test-Stanford.txt: Failed - {e}")

# Create benchmark DataFrame
benchmark_df = pd.DataFrame(all_samples)
benchmark_df = benchmark_df.rename(columns={"hashtag": "hashtags", "gold": "goldtruths"})

print()
print("=" * 60)
print(f"📊 Total benchmark samples: {len(benchmark_df)}")
print()
print("📈 Samples per dataset:")
print(benchmark_df["source"].value_counts().to_string())
print()

# Preview
print("📋 Sample entries:")
print(benchmark_df[["hashtags", "goldtruths", "source"]].head(15).to_string())
print()
print(f"✅ Benchmark dataset ready with {len(benchmark_df)} samples from {benchmark_df['source'].nunique()} datasets")


## 4. Execution Engine

Benchmark runner that measures latency and captures outputs for each segmenter.


In [ ]:
# ============================================================================
# CELL 4: Execution Engine
# ============================================================================

import time
from typing import Optional
from tqdm.auto import tqdm


def run_benchmark(
    segmenters: dict[str, Segmenter],
    dataset: pd.DataFrame,
    hashtag_column: str = "hashtags",
    limit: Optional[int] = None
) -> list[dict]:
    """
    Run benchmark across all segmenters on the given dataset.
    
    Args:
        segmenters: Dictionary mapping model names to Segmenter instances
        dataset: DataFrame containing hashtags to segment
        hashtag_column: Column name containing hashtags
        limit: Optional limit on number of samples to process
        
    Returns:
        List of dictionaries containing benchmark results
    """
    results = []
    
    # Get hashtags to process
    hashtags = dataset[hashtag_column].tolist()
    if limit:
        hashtags = hashtags[:limit]
    
    total_iterations = len(hashtags) * len(segmenters)
    
    with tqdm(total=total_iterations, desc="Benchmarking") as pbar:
        for hashtag in hashtags:
            for model_name, segmenter in segmenters.items():
                pbar.set_description(f"{model_name}: {hashtag[:20]}...")
                
                try:
                    # Measure latency
                    start_time = time.perf_counter()
                    output = segmenter.segment(hashtag)
                    end_time = time.perf_counter()
                    
                    latency_ms = (end_time - start_time) * 1000
                    error = None
                    
                except Exception as e:
                    output = f"ERROR: {str(e)[:50]}"
                    latency_ms = 0.0
                    error = str(e)
                
                results.append({
                    "input": hashtag,
                    "model": model_name,
                    "output": output,
                    "latency_ms": latency_ms,
                    "error": error
                })
                
                pbar.update(1)
    
    return results


def results_to_dataframe(results: list[dict]) -> pd.DataFrame:
    """Convert benchmark results to a pandas DataFrame."""
    return pd.DataFrame(results)


def create_comparison_table(results_df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a wide-format comparison table showing outputs side-by-side.
    
    Args:
        results_df: DataFrame with benchmark results
        
    Returns:
        Wide-format DataFrame with models as columns
    """
    # Pivot to get outputs side by side
    comparison = results_df.pivot(
        index="input",
        columns="model",
        values="output"
    ).reset_index()
    
    return comparison


print("✅ Execution engine ready!")


## 5. Initialize Segmenters

Create instances of all segmentation tools. We initialize fast models first, then heavier models.


In [ ]:
# ============================================================================
# CELL 5: Initialize Segmenters (Fast Models)
# ============================================================================

# Initialize fast, lightweight segmenters first
segmenters = {}

print("Initializing WordNinja...")
segmenters["WordNinja"] = WordNinjaSegmenter()

print("Initializing SymSpell...")
segmenters["SymSpell"] = SymSpellSegmenter()

print("Initializing Ekphrasis...")
segmenters["Ekphrasis"] = EkphrasisSegmenter()

print()
print("✅ Fast segmenters initialized!")
print(f"   Models ready: {list(segmenters.keys())}")


In [ ]:
# ============================================================================
# CELL 5b: Initialize Hashformers Models (No Reranker)
# ============================================================================

print("Initializing Hashformers (GPT-2 baseline)...")
segmenters["Hashformers-GPT2"] = HashformersSegmenter(
    segmenter_model="gpt2",
    segmenter_type="incremental",
    reranker_model=None,
    reranker_type=None
)

print("Initializing Hashformers (DistilGPT2)...")
segmenters["Hashformers-DistilGPT2"] = HashformersSegmenter(
    segmenter_model="distilgpt2",
    segmenter_type="incremental",
    reranker_model=None,
    reranker_type=None
)

print()
print("✅ Hashformers models initialized (no reranker)!")
print(f"   Models ready: {list(segmenters.keys())}")


In [ ]:
# ============================================================================
# CELL 5c: Initialize Local LLM (Optional - requires GPU)
# ============================================================================

# Clear GPU memory before loading LLM
clear_gpu_memory()

# Check if we have enough GPU memory
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {gpu_mem_gb:.1f} GB")
    
    if gpu_mem_gb >= 2:
        print("\nInitializing Local LLM (Qwen2-0.5B-Instruct, 4-bit quantized)...")
        print("This may take a few minutes...")
        
        try:
            segmenters["LLM-Qwen2"] = LocalLLMSegmenter(
                model_name="Qwen/Qwen2-0.5B-Instruct",
                load_in_4bit=True,
                max_new_tokens=64
            )
            print("✅ LLM initialized!")
        except Exception as e:
            print(f"⚠️ Could not load LLM: {e}")
            print("   Continuing without LLM segmenter...")
    else:
        print(f"⚠️ Insufficient GPU memory ({gpu_mem_gb:.1f} GB). Skipping LLM.")
        print("   Need at least 2 GB for 4-bit quantized Qwen2-0.5B.")
else:
    print("⚠️ No GPU available. Skipping LLM segmenter.")
    print("   Enable GPU runtime: Runtime > Change runtime type > T4 GPU")

print()
print(f"✅ All segmenters ready: {list(segmenters.keys())}")


## 6. Run Benchmark

Execute the benchmark across all segmenters.


In [ ]:
# ============================================================================
# CELL 6: Run Benchmark
# ============================================================================

print(f"Running benchmark on {len(benchmark_df)} hashtags...")
print(f"Models: {list(segmenters.keys())}")
print()

# Run the benchmark
results = run_benchmark(
    segmenters=segmenters,
    dataset=benchmark_df,
    hashtag_column="hashtags"
)

# Convert to DataFrame
results_df = results_to_dataframe(results)

print()
print(f"✅ Benchmark complete!")
print(f"   Total measurements: {len(results_df)}")

# Check for errors
errors = results_df[results_df["error"].notna()]
if len(errors) > 0:
    print(f"   ⚠️ Errors encountered: {len(errors)}")
else:
    print(f"   No errors encountered")


## 7. Analysis & Visualization

### 7.1 Side-by-Side Comparison Table

Compare segmentation outputs across all models.


In [ ]:
# ============================================================================
# CELL 7.1: Side-by-Side Comparison Table
# ============================================================================

# Create comparison table
comparison_df = create_comparison_table(results_df)

# Reorder columns for better readability
model_order = ["WordNinja", "SymSpell", "Ekphrasis", "Hashformers-GPT2", "Hashformers-DistilGPT2"]
if "LLM-Qwen2" in comparison_df.columns:
    model_order.append("LLM-Qwen2")

# Only include columns that exist
available_cols = ["input"] + [col for col in model_order if col in comparison_df.columns]
comparison_df = comparison_df[available_cols]

print("📊 Segmentation Output Comparison")
print("=" * 80)
print()

# Display with better formatting
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.max_rows', 50)

display(comparison_df.head(30))


### 7.2 Latency Analysis

Average segmentation time per model.


In [ ]:
# ============================================================================
# CELL 7.2: Latency Analysis
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

# Calculate latency statistics
latency_stats = results_df.groupby("model")["latency_ms"].agg(["mean", "std", "min", "max"]).reset_index()
latency_stats.columns = ["Model", "Mean (ms)", "Std (ms)", "Min (ms)", "Max (ms)"]
latency_stats = latency_stats.sort_values("Mean (ms)")

print("⏱️ Latency Statistics")
print("=" * 80)
print()
print(latency_stats.to_string(index=False))
print()

# Calculate hashtags per second (handle zero latency)
latency_stats["Hashtags/sec"] = latency_stats["Mean (ms)"].apply(
    lambda x: 1000 / x if x > 0 else 0
)
print("\n📈 Throughput (hashtags per second):")
for _, row in latency_stats.iterrows():
    if row["Hashtags/sec"] > 0:
        print(f"   {row['Model']:25s}: {row['Hashtags/sec']:8.2f} hashtags/sec")
    else:
        print(f"   {row['Model']:25s}: N/A (errors occurred)")


In [ ]:
# ============================================================================
# CELL 7.3: Latency Bar Chart
# ============================================================================

# Set up the plot style
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Color palette
colors = sns.color_palette("husl", len(segmenters))

# Plot 1: Average Latency (linear scale)
ax1 = axes[0]
order = latency_stats.sort_values("Mean (ms)")["Model"].tolist()
sns.barplot(
    data=results_df, 
    x="model", 
    y="latency_ms", 
    order=order,
    palette=colors,
    errorbar="sd",
    ax=ax1
)
ax1.set_xlabel("Model", fontsize=12)
ax1.set_ylabel("Latency (ms)", fontsize=12)
ax1.set_title("Average Segmentation Latency by Model", fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# Add value labels on bars
for i, p in enumerate(ax1.patches):
    ax1.annotate(
        f'{p.get_height():.1f}',
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center', va='bottom',
        fontsize=9, fontweight='bold'
    )

# Plot 2: Log scale for better comparison of fast vs slow models
ax2 = axes[1]
sns.barplot(
    data=results_df, 
    x="model", 
    y="latency_ms", 
    order=order,
    palette=colors,
    errorbar="sd",
    ax=ax2
)
ax2.set_yscale('log')
ax2.set_xlabel("Model", fontsize=12)
ax2.set_ylabel("Latency (ms) - Log Scale", fontsize=12)
ax2.set_title("Latency Comparison (Log Scale)", fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("benchmark_latency.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n💾 Chart saved to: benchmark_latency.png")


### 7.3 Accuracy Evaluation

Compute accuracy metrics against gold standard segmentations.


In [ ]:
# ============================================================================
# CELL 7.3: Accuracy Evaluation
# ============================================================================

from hashformers.evaluation.modeler import Modeler
import ast

def evaluate_model(predictions: list[str], gold_truths: list[str]) -> dict:
    """
    Evaluate predictions against gold standard using Modeler.
    
    Args:
        predictions: List of predicted segmentations
        gold_truths: List of gold standard segmentations (may contain multiple valid options)
        
    Returns:
        Dictionary with accuracy, precision, recall, and F1 score
    """
    modeler = Modeler()
    
    for pred, gold_str in zip(predictions, gold_truths):
        # Parse gold truths (stored as string representation of list)
        try:
            gold_options = ast.literal_eval(gold_str)
            if isinstance(gold_options, list):
                # Find best matching gold (or use first)
                gold = gold_options[0].lower().strip()
                # Check if any gold matches exactly
                pred_normalized = pred.lower().strip()
                for g in gold_options:
                    if g.lower().strip() == pred_normalized:
                        gold = g.lower().strip()
                        break
            else:
                gold = str(gold_options).lower().strip()
        except:
            gold = str(gold_str).lower().strip()
        
        pred_normalized = pred.lower().strip()
        modeler.countEntry(pred_normalized, gold)
    
    return {
        "accuracy": modeler.calculateAccuracy(),
        "precision": modeler.calculatePrecision(),
        "recall": modeler.calculateRecall(),
        "f1": modeler.calculateFScore()
    }

# Get gold truths for benchmark samples
gold_truths = benchmark_df["goldtruths"].tolist()

# Evaluate each model
print("📊 Accuracy Evaluation (against gold standard)")
print("=" * 80)
print()

evaluation_results = []
for model_name in segmenters.keys():
    # Get predictions for this model
    model_results = results_df[results_df["model"] == model_name]
    predictions = model_results["output"].tolist()
    
    # Compute metrics
    metrics = evaluate_model(predictions, gold_truths)
    metrics["model"] = model_name
    evaluation_results.append(metrics)
    
    print(f"📈 {model_name}:")
    print(f"   Accuracy:  {metrics['accuracy']:.2f}%")
    print(f"   Precision: {metrics['precision']:.2f}%")
    print(f"   Recall:    {metrics['recall']:.2f}%")
    print(f"   F1 Score:  {metrics['f1']:.2f}%")
    print()

# Create evaluation DataFrame
eval_df = pd.DataFrame(evaluation_results)
eval_df = eval_df[["model", "accuracy", "precision", "recall", "f1"]]
eval_df = eval_df.sort_values("accuracy", ascending=False)

print("\n📋 Summary Table:")
print(eval_df.to_string(index=False))


### 7.4 Qualitative Sample Analysis

Detailed look at interesting segmentation cases where models disagree.


In [ ]:
# ============================================================================
# CELL 7.4: Qualitative Sample Analysis
# ============================================================================

# Find cases where models disagree
def find_disagreements(comparison_df: pd.DataFrame) -> pd.DataFrame:
    """Find hashtags where models produced different outputs."""
    model_cols = [col for col in comparison_df.columns if col != "input"]
    
    disagreements = []
    for _, row in comparison_df.iterrows():
        outputs = [row[col] for col in model_cols if pd.notna(row[col])]
        # Normalize for comparison (lowercase, strip)
        normalized = [str(o).lower().strip() for o in outputs]
        if len(set(normalized)) > 1:  # Models disagree
            disagreements.append(row)
    
    return pd.DataFrame(disagreements)

disagreement_df = find_disagreements(comparison_df)

print("🔍 Cases Where Models Disagree")
print("=" * 80)
print(f"Found {len(disagreement_df)} hashtags with different segmentations")
print()

if len(disagreement_df) > 0:
    # Show first 15 disagreements
    display(disagreement_df.head(15))
else:
    print("All models produced identical outputs!")


## 8. Summary & Conclusions


In [ ]:
# ============================================================================
# CELL 8: Summary
# ============================================================================

print("=" * 80)
print("📊 BENCHMARK SUMMARY")
print("=" * 80)
print()

print("🔧 Models Benchmarked:")
for model in segmenters.keys():
    print(f"   • {model}")
print()

print(f"📝 Dataset: {benchmark_df['source'].nunique()} datasets ({len(benchmark_df)} total samples)")
print()

print("⏱️ Speed Ranking (fastest to slowest):")
speed_ranking = latency_stats.sort_values("Mean (ms)")
for i, (_, row) in enumerate(speed_ranking.iterrows(), 1):
    mean_ms = row["Mean (ms)"]
    if mean_ms > 0:
        throughput = 1000 / mean_ms
        print(f"   {i}. {row['Model']:25s} - {mean_ms:8.2f} ms ({throughput:.1f} hashtags/sec)")
    else:
        print(f"   {i}. {row['Model']:25s} - {mean_ms:8.2f} ms (N/A - errors occurred)")
print()

print("📈 Key Observations:")
print("   • Statistical methods (WordNinja, SymSpell, Ekphrasis) are significantly faster")
print("   • Hashformers provides better segmentation quality at the cost of speed")
print("   • LLM-based segmentation is slowest but can handle complex cases")
print()

print("💡 Recommendations:")
print("   • For high-throughput applications: Use WordNinja or Ekphrasis")
print("   • For best accuracy: Use Hashformers (consider adding reranker for production)")
print("   • For research/analysis: Consider the speed-accuracy tradeoff")
print()

print("=" * 80)
print("✅ Benchmark complete! Results saved to benchmark_latency.png")
print("=" * 80)


## 9. Export Results (Optional)

Save results to CSV for further analysis.


In [ ]:
# ============================================================================
# CELL 9: Export Results
# ============================================================================

# Save detailed results
results_df.to_csv("benchmark_results_detailed.csv", index=False)
print("💾 Detailed results saved to: benchmark_results_detailed.csv")

# Save comparison table
comparison_df.to_csv("benchmark_comparison.csv", index=False)
print("💾 Comparison table saved to: benchmark_comparison.csv")

# Save latency statistics
latency_stats.to_csv("benchmark_latency_stats.csv", index=False)
print("💾 Latency statistics saved to: benchmark_latency_stats.csv")

print()
print("📁 All results exported successfully!")
